# Spell and Rune Explorer

**Schema(s)** defining this data:
- `contracts/schemas/spells.schema.json` — **SpellsPack**: `spells[]` with spellId, name, type, categoryId, rarityId, manaCost, forgeCostManaCrystals, runeCombo, power, effectIds, durationTicks.
- `contracts/schemas/spell-evolution.schema.json` — **Evolution table**: runeCombo → resultSpellId, resultName, statBonuses, effectIds, isSummon, minLevel, minAffinityPerRune.
- `contracts/schemas/spell-forge-costs.schema.json` — Default/override forge costs by rarity.
- `contracts/schemas/runes.schema.json` — Rune lookup (ids, labels).

**Assets / packs** we load:
- `contracts/data/content_spells.json` — spell pack
- `contracts/data/content_spell_evolution.json` — evolution table
- `contracts/data/config_spell_forge_costs.json` — forge costs
- `contracts/data/lookup_runes.json` — runes

**What this tool does:** View spells (filter by category/rarity/type), rune combos, evolution paths; rebalance mana/power/forge cost; inspect behaviour (category, effectIds, evolution gates).

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "packages" / "engine").is_dir() and (ROOT.parent / "packages" / "engine").is_dir():
    ROOT = ROOT.parent
CONTRACTS_DIR = ROOT / "packages" / "engine" / "src" / "escape-the-dungeon" / "contracts"
DATA = CONTRACTS_DIR / "data"
SCHEMAS = CONTRACTS_DIR / "schemas"

SCHEMA_FILES = ["spells.schema.json", "spell-evolution.schema.json", "spell-forge-costs.schema.json", "runes.schema.json"]
PACK_FILES = ["content_spells.json", "content_spell_evolution.json", "config_spell_forge_costs.json", "lookup_runes.json"]

with open(DATA / "content_spells.json", encoding="utf-8") as f:
    SPELLS_PACK = json.load(f)
with open(DATA / "content_spell_evolution.json", encoding="utf-8") as f:
    EVOLUTION_PACK = json.load(f)
with open(DATA / "config_spell_forge_costs.json", encoding="utf-8") as f:
    FORGE_COSTS = json.load(f)
with open(DATA / "lookup_runes.json", encoding="utf-8") as f:
    RUNES_LOOKUP = json.load(f)

print("Schemas:", [str(SCHEMAS / s) for s in SCHEMA_FILES])
print("Packs:  ", [str(DATA / p) for p in PACK_FILES])
print(f"Loaded {len(SPELLS_PACK['spells'])} spells, {len(EVOLUTION_PACK['evolutionTable'])} evolution rows, {len(RUNES_LOOKUP.get('runes', []))} runes")

## View spells

Filter by category, rarity, or type. **Rebalance:** edit manaCost, power, forgeCostManaCrystals in `content_spells.json`. **Behaviour:** categoryId, effectIds, runeCombo.

In [ ]:
def view_spells(category=None, rarity=None, spell_type=None, max_rows=None):
    rows = SPELLS_PACK["spells"]
    if category:
        rows = [r for r in rows if r.get("categoryId") == category]
    if rarity:
        rows = [r for r in rows if r.get("rarityId") == rarity]
    if spell_type:
        rows = [r for r in rows if r.get("type") == spell_type]
    if max_rows is not None:
        rows = rows[:max_rows]
    for s in rows:
        mana = s.get("manaCost", "—")
        power = s.get("power", "—")
        forge = s.get("forgeCostManaCrystals", "—")
        combo = ",".join(s.get("runeCombo") or [])
        effects = ",".join(s.get("effectIds") or [])
        print(f"{s['spellId']:20} | {s['name']:18} | {s.get('categoryId',''):14} | {s.get('rarityId',''):8} | mana={mana} power={power} forge={forge} | runes=[{combo}] effects=[{effects}]")
    return rows

print("All spells (first 12):")
view_spells(max_rows=12)
print("\nBy categoryId=combat:")
view_spells(category="combat")

## View evolution table

**Rebalance:** minLevel, minAffinityPerRune, statBonuses. **Behaviour:** runeCombo → resultSpellId, isSummon.

In [ ]:
for e in EVOLUTION_PACK["evolutionTable"]:
    combo = ",".join(e.get("runeCombo") or [])
    print(f"{e['evolutionId']:20} | {combo:30} → {e.get('resultSpellId','')} ({e.get('resultName','')}) | summon={e.get('isSummon', False)} minLvl={e.get('minLevel','—')} minAff={e.get('minAffinityPerRune','—')}")

## View forge costs and runes

**Rebalance:** defaultByRarity and overrides in `config_spell_forge_costs.json`.

In [ ]:
print("Forge costs (default by rarity):", FORGE_COSTS.get("defaultByRarity", {}))
print("Overrides (spellId → crystals):", FORGE_COSTS.get("overrides", {}))
print("\nRunes:")
for r in RUNES_LOOKUP.get("runes", [])[:20]:
    print(f"  {r.get('runeId','')} {r.get('name','')}")
if len(RUNES_LOOKUP.get("runes", [])) > 20:
    print(f"  ... and {len(RUNES_LOOKUP['runes']) - 20} more")